# 01 · Data preprocessing

Launches the data step: `MILDataModule.setup()` reads the CSV, builds
**bags** (one bag = one molecule, instances = conformers), optionally clusters
conformers per molecule, fits a per-fingerprint-block `StandardScaler` on the
**train split only**, and produces the train/val/test datasets.

The split is predefined (`split` column: 0=train, 1=val, 2=test) — the same
`MILDataModule` is used for final training.

In [ ]:
# --- Bootstrap: make the notebook run from anywhere ---
import os, sys, logging
from pathlib import Path

# Locate the project root (folder that contains the `ppl` package).
here = Path.cwd()
PROJECT_ROOT = next(
    (p for p in [here, *here.parents] if (p / 'ppl' / '__init__.py').exists()),
    None,
)
if PROJECT_ROOT is None:
    # Fallback: this notebook lives in <root>/notebooks/
    PROJECT_ROOT = Path('__file__' in globals() and __file__ or '.').resolve().parent.parent

os.chdir(PROJECT_ROOT)                       # pipeline writes outputs relative to cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
print('Project root:', PROJECT_ROOT)

In [ ]:
# Path to the experiment YAML. Edit this to point at a different config.
CONFIG_PATH = PROJECT_ROOT / 'ppl/utils/experiment_configs/run_config.yaml'
assert CONFIG_PATH.exists(), f'Config not found: {CONFIG_PATH}'
print('Using config:', CONFIG_PATH.relative_to(PROJECT_ROOT))

## Build the data config

We reuse `PipelineConfigManager` so the `DataLoaderConfig` is identical to what
the pipeline builds (defaults + YAML overrides + experiment-name/cache wiring).

In [ ]:
from ppl.utils.modelling_configs.pipeline_config import PipelineConfig
from ppl.utils.pipeline.config_manager import PipelineConfigManager

cfg = PipelineConfig.from_yaml(CONFIG_PATH)
data_cfg = PipelineConfigManager(cfg).data_cfg
print('CSV      :', data_cfg.csv_path)
print('task     :', data_cfg.task)
print('batch    :', data_cfg.batch_size)
print('clustering:', data_cfg.cluster_instances)

## Run preprocessing

`setup()` does all the heavy lifting: it reads the CSV, applies the predefined
split (train=0, val=1, test=2), builds bags, clusters, and scales features.

In [ ]:
from ppl.utils.mil_data_handling.data_loader import MILDataModule

dm = MILDataModule(data_cfg)
dm.setup('fit')   # loads CSV, applies predefined split, builds bags, clusters, scales
print('setup complete')

## Inspect the result

In [ ]:
n_feat = len(dm.feature_names)
print('num descriptors (input_dim):', n_feat)
print('first 8 feature names      :', dm.feature_names[:8])
print()
for name, ds in [('train', dm._train), ('val', dm._val), ('test', dm._test)]:
    n = 0 if ds is None else len(ds)
    print(f'{name:<5} bags: {n}')

### Peek at one batch

The batch layout is `(bags, labels, bag_ids, padding_mask, cluster_ids, series_labels)`.
Shapes tell you the padded bag size and descriptor dimension.

In [ ]:
train_loader = dm.train_dataloader()
batch = next(iter(train_loader))

def describe(x):
    import torch
    if isinstance(x, torch.Tensor):
        return f'Tensor{tuple(x.shape)} {x.dtype}'
    if isinstance(x, (list, tuple)):
        return f'{type(x).__name__}(len={len(x)})'
    return type(x).__name__

labels = ['bags', 'labels', 'bag_ids', 'padding_mask', 'cluster_ids', 'series_labels']
for i, part in enumerate(batch):
    name = labels[i] if i < len(labels) else f'item_{i}'
    print(f'{name:<14}: {describe(part)}')

### Label distribution

Quick sanity check on the endpoint values across splits.

In [ ]:
import numpy as np
import torch

def collect_labels(loader):
    ys = []
    for b in loader:
        y = b[1]
        ys.append(y.detach().cpu().numpy().reshape(-1))
    return np.concatenate(ys) if ys else np.array([])

for name, loader in [('train', dm.train_dataloader()),
                     ('val', dm.val_dataloader()),
                     ('test', dm.test_dataloader())]:
    if loader is None or (hasattr(loader, '__len__') and len(loader) == 0):
        print(f'{name:<5}: (empty)')
        continue
    y = collect_labels(loader)
    if y.size:
        print(f'{name:<5}: n={y.size:<4} mean={y.mean():.3f} std={y.std():.3f} min={y.min():.3f} max={y.max():.3f}')

---
The key output for the next step is **`input_dim = len(dm.feature_names)`**.
Continue with **`02_model_construction.ipynb`**.